# Email-Based Job Approval (Extended) - End-to-End Test

Extends `test_email_approval.ipynb` to cover:

1. The `sync` service in `syft_bg` (required for email approval to see new jobs from DS).
2. **Manual approval**: DS submits a job → DO replies `approve` → job runs → DS gets results.
3. **Email auto-approval**: DS submits a job containing `params.json` → DO replies `auto-approve` → an auto-approval object is created. A follow-up job with the same `main.py` is approved without any email interaction.
4. **API-driven auto-approval**: turn a pending job into an auto-approval rule via `syft_bg.auto_approve_job(...)` and verify the rule is applied automatically.

### Prerequisites
- GCP project with **Gmail API** and **Pub/Sub API** enabled
- OAuth credentials JSON (`app_credentials.json`)
- DS token (`token_ds.json`)

## Configuration

Fill in your email addresses and the Project ID below.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

DO_EMAIL = "sameer@openmined.org"
DS_EMAIL = "som.wom.beach@gmail.com"
GCP_PROJECT_ID = "syft-client-dogfood"

# Credential paths (relative to notebooks/beach/internal)
APP_CREDENTIALS = Path("../../../credentials/app_credentials.json")
TOKEN_DS = Path("../../../credentials/token_ds.json")

assert APP_CREDENTIALS.exists(), "App credentials missing"
assert TOKEN_DS.exists(), "DS credentials missing"

## Step 1: Create DO Drive token from app credentials

This converts the OAuth app credentials to a user-authorized Drive token.
It will open a browser window for you to authorize.

In [3]:
import syft_client as sc

do_token_path = str(APP_CREDENTIALS.parent / "token_do_email_test.json")
# do_token_path = sc.credentials_to_token(
#     credentials_path=str(APP_CREDENTIALS),
#     output_path=do_token_path,
#     do_scopes=True
# )
# print(f"DO token saved to: {do_token_path}")

## Step 2: Login as Data Owner and Data Scientist

In [4]:
do_client = sc.login_do(email=DO_EMAIL, token_path=str(do_token_path))

Created user directory: /Users/swag/SyftBox_sameer@openmined.org/sameer@openmined.org
Ensured directories exist: /Users/swag/SyftBox_sameer@openmined.org/sameer@openmined.org/app_data/job
No checkpoints found, downloading all events...
✅ Connected to 0 peers


In [5]:
ds_client = sc.login_ds(email=DS_EMAIL, token_path=str(TOKEN_DS))

Created user directory: /Users/swag/SyftBox_som.wom.beach@gmail.com/som.wom.beach@gmail.com
✅ Connected to 0 peers


## Step 2a: Optional — wipe prior state

In [6]:
# do_client.delete_syftbox()
# ds_client.delete_syftbox()

## Step 3: Set up peer relationship

DS requests access to DO, then DO approves.

In [7]:
ds_client.add_peer(DO_EMAIL)

do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)

print("Peer relationship established")

🔄 Adding sameer@openmined.org on google_personal.gdrive_files...
✅ Added sameer@openmined.org to google_personal.gdrive_files
✅ Peer sameer@openmined.org added successfully on 1 transport(s)
• google_personal.gdrive_files
🔄 Adding som.wom.beach@gmail.com on google_personal.gdrive_files...
✅ Added som.wom.beach@gmail.com to google_personal.gdrive_files
Approved peer request from som.wom.beach@gmail.com
Peer relationship established


## Step 4: Set up `syft-bg`

In addition to `notify` and `email_approve`, we now also start the `sync` service — it keeps the DO-side SyftBox up to date with Drive so that new jobs submitted by DS are visible to the email-approval service. We also start the `approve` service, which is what applies the auto-approval objects to future jobs.

In [8]:
import syft_bg

In [9]:
syft_bg.reset()

Reset complete: stopped services, cleared state, config, logs, and PID files.


In [10]:
syft_bg.init(
    do_email=DO_EMAIL,
    token_path=do_token_path,
    settings={
        "email_approve": {"gcp_project_id": GCP_PROJECT_ID},
    },
)

Stored token at /Users/swag/.syft-bg/token.json
Config saved to /Users/swag/.syft-bg/config.yaml


In [11]:
# syft_bg.config

In [12]:
syft_bg.ensure_running(
    services=["sync", "notify", "email_approve", "approve"],
    restart=True,
)

sync started
notify started
email_approve started
approve started


In [13]:
syft_bg.status

syft-bg status
========================================
  email:       sameer@openmined.org
  syftbox:     /Users/swag/SyftBox_sameer@openmined.org
  environment: local

tokens
----------------------------------------
  notify.gmail_token_path             ready
  notify.drive_token_path             ready
  approve.drive_token_path            ready
  email_approve.gmail_token_path      ready

services
----------------------------------------
  notify           running (PID 55304) (not installed)
  approve          running (PID 55306) (not installed)
  email_approve    running (PID 55305) (not installed)
  sync             running (PID 55303) (not installed)

## Step 5: Create a dataset

In [14]:
do_client.create_dataset(
    name="testdata",
    mock_path="mock.txt",
    private_path="private.txt",
    users=[DS_EMAIL],
)

name,testdata
created_at,2026-04-28 17:32:28
summary,None
tags,[]
location,None


## Part A — Manual approval via email reply (`approve`)

DS submits a job. The `notify` service emails the DO. The DO replies `approve`. The `email_approve` service runs and shares the results back. The job uses: 
1. A dataset hosted by the data owner
2. A simple library dependence (Pandas)
3. Output written to a file

In [15]:
import tempfile
import uuid

MANUAL_JOB_NAME = f"email_test_{uuid.uuid4().hex[:8]}"
MANUAL_DIR = Path(tempfile.mkdtemp(prefix="email_approve_manual_"))
manual_job = MANUAL_DIR / "main.py"
manual_job.write_text('''import os, json
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata")
df = pd.read_fwf(resolved_path)

result = {"message": "Hello from email-approved job!", "value": df.columns[0]}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(manual_job),
    job_name=MANUAL_JOB_NAME,
    force_submission=True,
    dependencies=["pandas"]
)
print(f"Job '{MANUAL_JOB_NAME}' submitted to {DO_EMAIL}")

Created user directory: /Users/swag/SyftBox_som.wom.beach@gmail.com/sameer@openmined.org
Submitted python job, job files are in /Users/swag/SyftBox_som.wom.beach@gmail.com/sameer@openmined.org/app_data/job/inbox/som.wom.beach@gmail.com/email_test_4df35b41
Job 'email_test_4df35b41' submitted to sameer@openmined.org


### Reply `approve` to the notification email

1. Open the **DO's Gmail inbox**.
2. Find the notification email for the job above.
3. Reply with: `approve`.

To reject instead, reply `deny <reason>`. Re-run the cell below until the status flips to `done`.

In [17]:
print(f"Check {DO_EMAIL}'s inbox and reply 'approve' to the job notification.")
job = next((j for j in do_client.job_client.jobs if j.name == MANUAL_JOB_NAME), None)
print(f"job {job.status}" if job else "job not visible yet")

Check sameer@openmined.org's inbox and reply 'approve' to the job notification.
job running


### DS fetches the result

In [19]:
ds_job = next(j for j in ds_client.jobs if j.name == MANUAL_JOB_NAME)
assert ds_job.status == "done", f"expected 'done', got {ds_job.status}"
print(f"Job {ds_job.name}: {ds_job.status}")
print(ds_job.output_paths[0].read_text())

Job email_test_4df35b41: done
{
  "message": "Hello from email-approved job!",
  "value": "private"
}


## Part B — Email auto-approval via `auto-approve` reply

The DS submits a job whose `code_path` is a folder with both `main.py` and `params.json`. The DO replies `auto-approve` to the notification email. This approves the current job **and** creates an auto-approval object so that future jobs with the same `main.py` (and differing `params.json`) are approved automatically.

Reference: `tests/unit/syft_bg/test_email_auto_approve_flow.py::test_email_auto_approve_creates_object_and_approves_future_jobs`.

In [20]:
import json

def create_parameterized_job(main_file: str, params: dict) -> Path:
    project_dir = Path(tempfile.mkdtemp(prefix="email_auto_approve_"))
    (project_dir / main_file).write_text('''import os, json

with open("params.json") as f:
    params = json.load(f)

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump({"params": params, "status": "done"}, f)
'''
    )
    (project_dir / "params.json").write_text(json.dumps(params))
    return project_dir

In [21]:
FIRST_JOB_NAME = f"auto_approve_{uuid.uuid4().hex[:8]}"
first_job = create_parameterized_job(main_file="main.py", params={"run": 1})

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(first_job),
    job_name=FIRST_JOB_NAME,
    entrypoint="main.py",
    force_submission=True,
)
print(f"Submitted {FIRST_JOB_NAME} (with params.json)")

Submitted python job, job files are in /Users/swag/SyftBox_som.wom.beach@gmail.com/sameer@openmined.org/app_data/job/inbox/som.wom.beach@gmail.com/auto_approve_e22d57b0
Submitted auto_approve_e22d57b0 (with params.json)


### Reply `auto-approve` to the notification email

Find the new notification for `FIRST_JOB` in the DO inbox and reply with: `auto-approve`.

Re-run the cell below until `status` is `done`.

In [22]:
job = next((j for j in do_client.job_client.jobs if j.name == FIRST_JOB_NAME), None)
print(f"job {job.status}" if job else "job not visible yet")

job done


### Verify the auto-approval object was created

In [48]:
from syft_bg.approve.config import AutoApproveConfig

cfg = AutoApproveConfig.load()
auto_obj = cfg.auto_approvals.objects[FIRST_JOB_NAME]

content_names = {e.relative_path for e in auto_obj.file_contents}
assert content_names == {"main.py"}, content_names
print(f"Auto-approval object '{FIRST_JOB_NAME}' created: {auto_obj}")

Auto-approval object 'auto_approve_e22d57b0' created: file_contents=[FileEntry(relative_path='main.py', path='/Users/swag/.syft-bg/auto_approvals/auto_approve_e22d57b0/main.py', hash='sha256:c0d635e1fafe4cb4fe9e1fce570a2d47e8d8b950bb7ea49e811f1536aa8668cf')] file_paths=['params.json'] peers=['som.wom.beach@gmail.com']


### DS fetches results for the first job

In [27]:
ds_first = next(j for j in ds_client.jobs if j.name == FIRST_JOB_NAME)
assert ds_first.status == "done", ds_first.status
result = json.loads(ds_first.output_paths[0].read_text())
print(f"DS received first-job result: {result}")

AssertionError: pending

### Submit a follow-up job that should be auto-approved

Same `main.py` (content-matched via hash), different `params.json` (name-matched only). No email interaction is required; the running `approve` service should handle this on its next tick.

In [28]:
SECOND_JOB_NAME = f"auto_approve_{uuid.uuid4().hex[:8]}"
second_job = create_parameterized_job(main_file="main.py", params={"run": 2})

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(second_job),
    job_name=SECOND_JOB_NAME,
    entrypoint="main.py",
    force_submission=True,
)
print(f"Submitted {SECOND_JOB_NAME}")

Submitted python job, job files are in /Users/swag/SyftBox_som.wom.beach@gmail.com/sameer@openmined.org/app_data/job/inbox/som.wom.beach@gmail.com/auto_approve_83dc771b
Submitted auto_approve_83dc771b


In [33]:
job = next((j for j in do_client.job_client.jobs if j.name == SECOND_JOB_NAME), None)
print(f"job {job.status}" if job else "job not visible yet")

job done


In [35]:
ds_second = next(j for j in ds_client.jobs if j.name == SECOND_JOB_NAME)
assert ds_second.status == "done", ds_second.status
result_2 = json.loads(ds_second.output_paths[0].read_text())
print(f"DS received second-job result (auto-approved): {result_2}")

DS received second-job result (auto-approved): {'params': {'run': 2}, 'status': 'done'}


## Part C — Create an auto-approval object manually via the API

Instead of replying `auto-approve` to an email, the DO can register an auto-approval object directly with `syft_bg.auto_approve_job(...)`. This takes a pending job as a template: content-matches the user files by default, name-matches whatever we pass as `file_paths`, and defaults `peers` to the submitter. The pending job itself is then auto-approved by the `approve` service on its next tick.

In [36]:
API_JOB_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"
api_job = create_parameterized_job(main_file="code.py", params={"run": "api-run"})

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(api_job),
    job_name=API_JOB_NAME,
    entrypoint="code.py",
    force_submission=True,
)
print(f"Submitted {API_JOB_NAME} (pending, awaiting DO-side auto_approve_job call)")

Submitted python job, job files are in /Users/swag/SyftBox_som.wom.beach@gmail.com/sameer@openmined.org/app_data/job/inbox/som.wom.beach@gmail.com/api_auto_81dd45c8
Submitted api_auto_81dd45c8 (pending, awaiting DO-side auto_approve_job call)


In [38]:
pending = [j for j in do_client.jobs if j.name == API_JOB_NAME]
assert pending, f"DO does not yet see {API_JOB_NAME} — wait for the sync service to tick and re-run"
assert pending[0].status == "pending", pending[0].status

### Manually approve all jobs like the one above

In [39]:
result = syft_bg.auto_approve_job(pending[0], file_paths=["params.json"])
assert result.success, result.error
print(result)

success=True name='api_auto_81dd45c8' file_contents=['code.py'] file_paths=['params.json'] peers=['som.wom.beach@gmail.com'] error=None


### Verify auto-approve config was added and job approved

In [42]:
cfg = AutoApproveConfig.load()
assert API_JOB_NAME in cfg.auto_approvals.objects, "auto_approve_job did not register a rule under the job's name"
api_obj = cfg.auto_approvals.objects[API_JOB_NAME]
assert {e.relative_path for e in api_obj.file_contents} == {"code.py"}
print(f"Rule '{API_JOB_NAME}' registered from the pending job: {api_obj}")

Rule 'api_auto_81dd45c8' registered from the pending job: file_contents=[FileEntry(relative_path='code.py', path='/Users/swag/.syft-bg/auto_approvals/api_auto_81dd45c8/code.py', hash='sha256:c0d635e1fafe4cb4fe9e1fce570a2d47e8d8b950bb7ea49e811f1536aa8668cf')] file_paths=['params.json'] peers=['som.wom.beach@gmail.com']


In [43]:
job = next((j for j in do_client.job_client.jobs if j.name == API_JOB_NAME), None)
print(f"job {job.status}" if job else "job not visible yet")

job done


In [44]:
ds_api = next(j for j in ds_client.jobs if j.name == API_JOB_NAME)
assert ds_api.status == "done", ds_api.status
api_result = json.loads(ds_api.output_paths[0].read_text())
print(f"DS received API-auto-approved job result: {api_result}")

DS received API-auto-approved job result: {'params': {'run': 'api-run'}, 'status': 'done'}


## Debugging: service logs

In [46]:
for svc in ("sync", "notify", "email_approve"): # Also "approve"
    print(f"\n=== {svc} ===")
    syft_bg.logs(svc, n=20)


=== sync ===
[SyncOrchestrator] Cycle 36 completed in 4869ms (2 jobs, 1 peers)
[SyncOrchestrator] Cycle 37 completed in 5129ms (2 jobs, 1 peers)
[SyncOrchestrator] Cycle 38 completed in 4945ms (2 jobs, 1 peers)
[SyncOrchestrator] Cycle 39 completed in 4995ms (2 jobs, 1 peers)
[SyncOrchestrator] Cycle 40 completed in 21914ms (2 jobs, 1 peers)
[SyncOrchestrator] Cycle 41 completed in 16883ms (3 jobs, 1 peers)
[SyncOrchestrator] Cycle 42 completed in 10821ms (3 jobs, 1 peers)
[SyncOrchestrator] Cycle 43 completed in 7512ms (3 jobs, 1 peers)
[SyncOrchestrator] Cycle 44 completed in 16214ms (4 jobs, 1 peers)
[SyncOrchestrator] Cycle 45 completed in 27428ms (4 jobs, 1 peers)
[SyncOrchestrator] Cycle 46 completed in 8660ms (4 jobs, 1 peers)
[SyncOrchestrator] Cycle 47 completed in 35358ms (4 jobs, 1 peers)
[SyncOrchestrator] Cycle 48 completed in 17751ms (4 jobs, 1 peers)
[SyncOrchestrator] Cycle 49 completed in 10352ms (4 jobs, 1 peers)
[SyncOrchestrator] Cycle 50 completed in 10142ms (4 jo

## Cleanup

In [47]:
# syft_bg.stop()